# Quick check of the `padfix` code (about 5–10 minutes)

**How to run:** in Settings, set Accelerator to **GPU T4** and make sure **Internet is on** (the same settings as your experiments). Then click **Run All**.

The notebook finds COCO by itself. If it isn't attached, it downloads the COCO val2017 images (~800 MB, a few minutes), just like the experiment notebooks did.

**What you should see at the end:**
- default padded batches: about **30–40%** of images change;
- with `self`: about **0%**;
- exact-shape buckets: about **0%**.

In [ ]:
import os
os.makedirs('padfix', exist_ok=True)
print('ok')

In [ ]:
%%writefile padfix/__init__.py
from .self_fix import Fix, enable_self_torchvision, enable_self_hf
from .stability import compare, change_rate, greedy_match, iou_matrix
from .buckets import exact_shape_buckets, torchvision_canvas

__all__ = ["Fix", "enable_self_torchvision", "enable_self_hf", "compare", "change_rate",
           "greedy_match", "iou_matrix", "exact_shape_buckets", "torchvision_canvas"]


In [ ]:
%%writefile padfix/self_fix.py
"""`self`: own-canvas masking for batch-invariant object detection.

Every image in a padded batch reproduces its own batch-1 computation (paper, Sect. 6).
The code between the CORE markers is copied verbatim from the notebooks that produced
every number in the paper (`notebooks/final_fullval_group*.ipynb`, cell "The `self` fix").
The wrappers below it (`enable_self_torchvision`, `enable_self_hf`) only call
`Fix.begin` / `Fix.end` around each forward pass, as the notebooks' detector classes do.
"""
import math
import torch
import torch.nn as nn

# ----------------------------------------------------------------------------- CORE (verbatim)
class Fix:
    COMPONENTS = ("conv", "norm", "own", "exactmask")
    def __init__(self):
        for c in self.COMPONENTS:
            setattr(self, c, False)
        self.valid, self.canvas, self.trivial, self._cache = None, None, True, {}

    def set(self, components):
        for c in self.COMPONENTS:
            setattr(self, c, c in components)

    def any_on(self):
        return any(getattr(self, c) for c in self.COMPONENTS)

    def begin(self, valid, canvas):
        self.valid = [tuple(int(v) for v in s) for s in valid]
        self.canvas = tuple(int(c) for c in canvas)
        self.trivial = all(v == self.canvas for v in self.valid)

    def end(self):
        self.valid = None

    def active(self):
        return self.valid is not None and not self.trivial

    def applies(self, x):
        return self.active() and x.dim() == 4 and x.shape[0] == len(self.valid)

    @staticmethod
    def _stride(n, N):
        for s in (1, 2, 4, 8, 16, 32, 64, 128, 256):
            if math.ceil(N / s) == n:
                return s
        return N / n

    def extents(self, H, W):
        """each image's valid (rows, cols) at a feature map of size H x W"""
        sh, sw = self._stride(H, self.canvas[0]), self._stride(W, self.canvas[1])
        return [(math.ceil(h / sh), math.ceil(w / sw)) for h, w in self.valid]

    def mask(self, x):
        H, W = x.shape[-2:]
        key = (H, W, tuple(self.valid), self.canvas, x.device, x.dtype)
        if key not in self._cache:
            m = torch.zeros(len(self.valid), 1, H, W, device=x.device, dtype=x.dtype)
            for i, (h, w) in enumerate(self.extents(H, W)):
                m[i, :, :h, :w] = 1
            if len(self._cache) > 64:
                self._cache.clear()
            self._cache[key] = m
        return self._cache[key]

def _mask_conv_inputs(root, fix):
    """Zero the padding at the input of every conv / max-pool: each layer then sees exactly the zeros its
    batch-1 run sees past the canvas edge (activations are >= 0 after ReLU, so max-pool is unaffected)."""
    def pre(mod, inp):
        x = inp[0]
        if fix.conv and fix.applies(x):
            return (x * fix.mask(x),) + tuple(inp[1:])
    for m in root.modules():
        if isinstance(m, (nn.Conv2d, nn.MaxPool2d)):
            m.register_forward_pre_hook(pre)

def _masked_groupnorm(root, fix):
    """GroupNorm statistics over each image's own valid area only."""
    n = 0
    for mod in root.modules():
        if isinstance(mod, nn.GroupNorm):
            orig = mod.forward
            def fwd(x, mod=mod, orig=orig):
                if not (fix.norm and fix.applies(x)):
                    return orig(x)
                m = fix.mask(x).float(); B, C, H, W = x.shape; G = mod.num_groups
                xg, mg = x.float().reshape(B, G, C // G, H, W), m.reshape(B, 1, 1, H, W)
                cnt = mg.sum(dim=(2, 3, 4), keepdim=True) * (C // G)
                mean = (xg * mg).sum(dim=(2, 3, 4), keepdim=True) / cnt
                var = (((xg - mean) ** 2) * mg).sum(dim=(2, 3, 4), keepdim=True) / cnt
                y = ((xg - mean) / torch.sqrt(var + mod.eps)).reshape(B, C, H, W)
                if mod.affine:
                    y = y * mod.weight.float()[None, :, None, None] + mod.bias.float()[None, :, None, None]
                return y.to(x.dtype)
            mod.forward = fwd; n += 1
    return n

def _suppress_padding_scores(conv, fix):
    """Classification logits at positions that do not exist in the image's batch-1 run -> -1e4."""
    def hook(mod, inp, out):
        if fix.conv and fix.applies(out):
            return out.masked_fill(fix.mask(out) == 0, -1e4)
    conv.register_forward_hook(hook)

def _own_anchors(ag, fix):
    """Each image gets the anchors its own batch-1 canvas would give: torchvision spaces anchors at
    canvas // grid, so we use the image's own canvas and grid. Anchors at positions that do not exist in
    its batch-1 run become zero-size boxes, which torchvision removes."""
    orig = ag.forward
    def fwd(image_list, feature_maps):
        if not (fix.own and fix.active()):
            return orig(image_list, feature_maps)
        grid_sizes = [list(f.shape[-2:]) for f in feature_maps]
        Hc, Wc = image_list.tensors.shape[-2:]
        dtype, device = feature_maps[0].dtype, feature_maps[0].device
        ag.set_cell_anchors(dtype, device)
        n_cell = [ca.shape[0] for ca in ag.cell_anchors]
        out = []
        for ch, cw in fix.valid:
            strides, keep = [], []
            for (gh, gw), a in zip(grid_sizes, n_cell):
                sh, sw = 2 ** round(math.log2(Hc / gh)), 2 ** round(math.log2(Wc / gw))
                oh, ow = math.ceil(ch / sh), math.ceil(cw / sw)
                strides.append([torch.tensor(ch // oh, dtype=torch.int64, device=device),
                                torch.tensor(cw // ow, dtype=torch.int64, device=device)])
                k = torch.zeros(gh, gw, dtype=dtype, device=device); k[:oh, :ow] = 1
                keep.append(k.reshape(-1).repeat_interleave(a))
            anc = ag.grid_anchors(grid_sizes, strides)
            out.append(torch.cat([x * k[:, None] for x, k in zip(anc, keep)]))
        return out
    ag.forward = fwd

def install_fix_tv(model, fix):
    heads = [model.rpn.head] if hasattr(model, "rpn") else [model.head]
    for root in [model.backbone] + heads:
        _mask_conv_inputs(root, fix)
    has_roi = hasattr(model, "roi_heads")
    def zero_outputs(mod, inp, out):
        if not (fix.conv and fix.active()):
            return
        new = type(out)()
        for k, v in out.items():
            v = v * fix.mask(v)
            if fix.own and has_roi:
                # RoIAlign at batch 1 clamps samples at the canvas edge to the last row/column; copying that
                # row/column one step outward reproduces this inside a bigger canvas.
                H, W = v.shape[-2:]
                for i, (hl, wl) in enumerate(fix.extents(H, W)):
                    if wl < W:
                        v[i, :, :hl, wl] = v[i, :, :hl, wl - 1]
                    if hl < H:
                        v[i, :, hl, :min(wl + 1, W)] = v[i, :, hl - 1, :min(wl + 1, W)]
            new[k] = v
        return new
    model.backbone.register_forward_hook(zero_outputs)
    g = sum(_masked_groupnorm(h, fix) for h in heads) + _masked_groupnorm(model.backbone, fix)
    if hasattr(model, "rpn"):
        _own_anchors(model.rpn.anchor_generator, fix); _suppress_padding_scores(model.rpn.head.cls_logits, fix)
    else:
        _own_anchors(model.anchor_generator, fix); _suppress_padding_scores(model.head.classification_head.cls_logits, fix)
    fix.SELF = ("conv", "norm", "own") if g else ("conv", "own")

def install_fix_hf(model, fix):
    enc = model.model.backbone
    _mask_conv_inputs(enc, fix)
    orig = enc.forward
    def fwd(pixel_values, pixel_mask):
        out = orig(pixel_values, pixel_mask)
        if not (fix.exactmask and fix.active()):
            return out
        # the transformer's padding mask with exact extents (the library shrinks it with nearest-neighbour rounding)
        return [(fm, fix.mask(fm)[:, 0].bool()) for fm, _ in out]
    enc.forward = fwd
    fix.SELF = ("conv", "exactmask")

# ----------------------------------------------------------------------------- end of CORE


def _roundup(x, d):
    return int(math.ceil(x / d) * d)


def enable_self_torchvision(model):
    """Install `self` on a torchvision detector (Faster/Mask/Keypoint R-CNN, RetinaNet, FCOS).

    The model keeps its normal call signature, model(list_of_image_tensors). Inside a padded batch,
    every image then gets its batch-1 output (up to floating-point summation order).
    Returns the `Fix` object; `fix.set(())` turns the fix off, `fix.set(fix.SELF)` back on.
    """
    fix = Fix()
    install_fix_tv(model, fix)
    fix.set(fix.SELF)
    transform = model.transform
    orig_transform_forward = transform.forward

    def transform_forward(images, targets=None):
        image_list, targets = orig_transform_forward(images, targets)
        if fix.any_on():
            d = transform.size_divisible
            sizes = [tuple(int(v) for v in s) for s in image_list.image_sizes]
            valid = [(_roundup(h, d), _roundup(w, d)) for h, w in sizes] if fix.own else sizes
            fix.begin(valid, tuple(image_list.tensors.shape[-2:]))
        return image_list, targets

    transform.forward = transform_forward
    orig_forward = model.forward

    def forward(*args, **kwargs):
        try:
            return orig_forward(*args, **kwargs)
        finally:
            fix.end()

    model.forward = forward
    return fix


def enable_self_hf(model):
    """Install `self` on a Hugging Face DETR-family detector that takes a `pixel_mask`
    (tested: DETR, Conditional DETR). Call the model as usual with pixel_values and pixel_mask."""
    fix = Fix()
    install_fix_hf(model, fix)
    fix.set(fix.SELF)
    orig_forward = model.forward

    def forward(pixel_values=None, pixel_mask=None, **kwargs):
        if pixel_mask is not None and fix.any_on():
            valid = [(int(pixel_mask[i, :, 0].sum()), int(pixel_mask[i, 0, :].sum())) for i in range(pixel_mask.shape[0])]
            fix.begin(valid, tuple(pixel_values.shape[-2:]))
        try:
            return orig_forward(pixel_values=pixel_values, pixel_mask=pixel_mask, **kwargs)
        finally:
            fix.end()

    model.forward = forward
    return fix


In [ ]:
%%writefile padfix/stability.py
"""Per-image stability metrics (paper, Sect. 3.4), copied from the experiment notebooks.

A prediction is a dict with numpy arrays: boxes (N,4, xyxy), scores (N,), labels (N,).
"""
import numpy as np


def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def greedy_match(p, q, thr, iou_min=0.5):
    """Greedy, score-ordered, same-class matching of p's and q's detections with score >= thr."""
    ia = np.where(p["scores"] >= thr)[0]; ib = np.where(q["scores"] >= thr)[0]
    ia = ia[np.argsort(-p["scores"][ia])]
    M = iou_matrix(p["boxes"][ia], q["boxes"][ib]); used = set(); pairs = []
    for r, i in enumerate(ia):
        best, bc = -1.0, None
        for c, j in enumerate(ib):
            if c not in used and p["labels"][i] == q["labels"][j] and M[r, c] > best:
                best, bc = M[r, c], c
        if bc is not None and best >= iou_min:
            used.add(bc); pairs.append((i, ib[bc], best))
    return pairs, ia, ib


def compare(ref, new, score_thr=0.5, moved_iou=0.9):
    """ref: batch-1 prediction, new: batched prediction of the same image.

    Returns whether the image changed ("flip": any confident detection moved, lost or new),
    the per-detection counts, and the largest score change among matched detections (score >= 0.05).
    """
    pairs, ia, ib = greedy_match(ref, new, score_thr)
    stable = sum(iou >= moved_iou for _, _, iou in pairs)
    moved = len(pairs) - stable
    lost, gained = len(ia) - len(pairs), len(ib) - len(pairs)
    allp, _, _ = greedy_match(ref, new, 0.05)
    dscore = max([abs(ref["scores"][i] - new["scores"][j]) for i, j, _ in allp], default=0.0)
    return {"flip": (moved + lost + gained) > 0, "stable": stable, "moved": moved, "lost": lost,
            "gained": gained, "max_score_delta": dscore}


def change_rate(refs, news, **kw):
    """Share of images whose confident detections change (paper: "images changed")."""
    flips = [compare(r, n, **kw)["flip"] for r, n in zip(refs, news)]
    return float(np.mean(flips)) if flips else 0.0


In [ ]:
%%writefile padfix/buckets.py
"""Exact-shape bucketing (paper, Sect. 6.3): batch only images whose batch-1 canvases are identical."""
import math


def exact_shape_buckets(canvas_sizes, batch_size=8):
    """canvas_sizes: list of each image's batch-1 canvas (H, W), i.e. its model-input size rounded up
    to the library's size divisor (32 for torchvision, 1 for Hugging Face DETRs).
    Returns a list of batches (lists of indices) in which no image is padded beyond its own canvas."""
    groups = {}
    for i, s in enumerate(canvas_sizes):
        groups.setdefault(tuple(int(v) for v in s), []).append(i)
    return [g[k:k + batch_size] for g in groups.values() for k in range(0, len(g), batch_size)]


def torchvision_canvas(model, image_tensor):
    """Batch-1 canvas of one image for a torchvision detector (resize, then round up to size_divisible)."""
    import torch
    with torch.inference_mode():
        il, _ = model.transform([image_tensor])
    h, w = il.image_sizes[0]
    d = model.transform.size_divisible
    return (int(math.ceil(h / d) * d), int(math.ceil(w / d) * d))


In [ ]:
import glob, os, urllib.request, zipfile
# 1) COCO already attached?  2) otherwise download val2017 (same source as the experiment notebooks)
probe = glob.glob("/kaggle/input/**/000000000139.jpg", recursive=True)
if probe:
    IMAGES = os.path.dirname(probe[0])
else:
    D = "/kaggle/working/coco"; os.makedirs(D, exist_ok=True)
    IMAGES = os.path.join(D, "val2017")
    if not (os.path.isdir(IMAGES) and len(os.listdir(IMAGES)) >= 5000):
        zp = os.path.join(D, "val2017.zip")
        print("downloading COCO val2017 images (~800 MB) ...")
        urllib.request.urlretrieve("http://images.cocodataset.org/zips/val2017.zip", zp)
        with zipfile.ZipFile(zp) as z:
            z.extractall(D)
        os.remove(zp)
print("using", IMAGES, "-", len(glob.glob(os.path.join(IMAGES, "*.jpg"))), "images")

In [ ]:
import argparse, glob, os, random, sys
import numpy as np, torch, torchvision
from PIL import Image

sys.path.insert(0, ".")
from padfix import enable_self_torchvision, change_rate, exact_shape_buckets, torchvision_canvas

class args:
    images, n, batch = IMAGES, 64, 8

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
dev = "cuda" if torch.cuda.is_available() else "cpu"
W = torchvision.models.detection.FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights=W).to(dev).eval()
tf = W.transforms()
fix = enable_self_torchvision(model)

paths = sorted(glob.glob(os.path.join(args.images, "*.jpg")))[: args.n]
imgs = [tf(Image.open(p).convert("RGB")).to(dev) for p in paths]

def pack(o):
    return {k: o[k].float().cpu().numpy() for k in ("boxes", "scores", "labels")}

@torch.inference_mode()
def run(batches):
    out = [None] * len(imgs)
    for b in batches:
        for i, o in zip(b, model([imgs[i] for i in b])):
            out[i] = pack(o)
    return out

fix.set(())                                   # fix off
ref = run([[i] for i in range(len(imgs))])
order = list(range(len(imgs))); random.Random(0).shuffle(order)
rand = [order[k:k + args.batch] for k in range(0, len(order), args.batch)]
print(f"random batches of {args.batch}, default : {100 * change_rate(ref, run(rand)):.1f}% of images change")
fix.set(fix.SELF)                             # fix on
print(f"random batches of {args.batch}, + self  : {100 * change_rate(ref, run(rand)):.1f}% of images change")
fix.set(())
canv = [torchvision_canvas(model, x) for x in imgs]
print(f"exact-shape buckets           : {100 * change_rate(ref, run(exact_shape_buckets(canv, args.batch))):.1f}% of images change")